persiapan

In [1]:
import os
import shutil
import time
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum, count, window

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas10_Streaming").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Konfigurasi shuffle partitions sesuai instruksi
spark.conf.set("spark.sql.shuffle.partitions", "4")

26/11/05 13:35:17 WARN Utils: Your hostname, azizi-Aspire-A314-36M resolves to a loopback address: 127.0.1.1; using 10.90.66.50 instead (on interface wlp0s20f3)
26/11/05 13:35:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 13:35:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/05 13:35:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


A. Menyiapkan Streaming Source
Bagian ini mendefinisikan skema secara eksplisit dan membuat objek readStream dari folder tugas10_stream.

In [2]:
folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Definisi skema eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Definisi readStream
df_stream = spark.readStream \
    .schema(skema_pesanan) \
    .csv(folder_stream, header=False)

B. Agregasi per Cabang
Menambahkan kolom turunan total_harga dan melakukan agregasi berdasarkan

In [3]:
# Tambahkan kolom turunan total_harga
df_stream_with_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi per cabang
df_agg_cabang = df_stream_with_total.groupBy("nama_cabang") \
    .agg(
        sum("total_harga").alias("total_pendapatan"),
        count("pesanan_id").alias("jumlah_pesanan")
    )

# Jalankan streaming query dengan sink memory
query_cabang = df_agg_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

C. Agregasi Berbasis Window Waktu
Mendefinisikan ulang readStream dan mengelompokkan data berdasarkan window waktu 2 menit pada kolom waktu_pesan.

In [4]:
# Definisi ulang readStream untuk query kedua
df_stream2 = spark.readStream \
    .schema(skema_pesanan) \
    .csv(folder_stream, header=False)

df_stream2_with_total = df_stream2.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berbasis window 2 menit
df_agg_waktu = df_stream2_with_total.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    sum("total_harga").alias("total_pendapatan_window")
)

# Jalankan streaming query kedua
query_waktu = df_agg_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

D. Menguji dengan Data Simulasi
Mengirimkan 3 batch data CSV menggunakan fungsi rekursif dan jeda time.sleep() untuk membuktikan bahwa agregasi terakumulasi dengan benar.

In [5]:
def kirim_data_batch(nama_file, isi_data):
    with open(f"{folder_stream}/{nama_file}", "w") as f:
        f.write(isi_data)
    time.sleep(5) # Jeda agar Spark memiliki waktu untuk memproses data

# Batch 1
kirim_data_batch("batch1.csv", 
    "P01,Cabang_A,Kopi Susu,2,15000,2026-11-05 10:00:00\n"
    "P02,Cabang_B,Americano,1,20000,2026-11-05 10:01:00\n"
)
print("--- Hasil Agregasi Setelah Batch 1 ---")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

# Batch 2
kirim_data_batch("batch2.csv", 
    "P03,Cabang_A,Latte,1,25000,2026-11-05 10:02:00\n"
    "P04,Cabang_C,Espresso,2,18000,2026-11-05 10:02:30\n"
)
print("--- Hasil Agregasi Setelah Batch 2 ---")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

# Batch 3
kirim_data_batch("batch3.csv", 
    "P05,Cabang_B,Kopi Susu,3,15000,2026-11-05 10:03:00\n"
    "P06,Cabang_A,Americano,1,20000,2026-11-05 10:04:00\n"
)
print("--- Hasil Agregasi Setelah Batch 3 ---")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY nama_cabang").show()

--- Hasil Agregasi Setelah Batch 1 ---
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Cabang_A|           30000|             1|
|   Cabang_B|           20000|             1|
+-----------+----------------+--------------+

--- Hasil Agregasi Setelah Batch 2 ---
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Cabang_A|           55000|             2|
|   Cabang_B|           20000|             1|
|   Cabang_C|           36000|             1|
+-----------+----------------+--------------+

--- Hasil Agregasi Setelah Batch 3 ---
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Cabang_A|           75000|             3|
|   Cabang_B|           65000|             2|
|   Cabang_C|           36000|             1|
+------

E. Sink Berbasis Berkas (Tantangan)
Menulis DataFrame mentah secara berkelanjutan ke dalam format Parquet dan membuktikannya.

In [6]:
# Hapus folder output jika sudah ada dari percobaan sebelumnya
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# Membuat query ketiga untuk menulis DataFrame mentah
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# Kirim batch data baru khusus untuk menguji Parquet
kirim_data_batch("batch_parquet.csv", "P07,Cabang_C,Kopi Susu,2,15000,2026-11-05 10:05:00\n")
time.sleep(8) # Jeda ekstra untuk memastikan penulisan file selesai

In [7]:
!ls tugas10_output

part-00000-5fe1908c-7ad8-49bf-87f1-b409050b7995-c000.snappy.parquet
part-00001-37bb6c93-5252-49d4-ac32-32bbb8dc4803-c000.snappy.parquet
part-00002-6b845253-fbda-4df5-af0c-6cde3d7fec65-c000.snappy.parquet
part-00003-b6ba6b35-0b57-4d8a-b8a5-fba8e040bb0e-c000.snappy.parquet
_spark_metadata


In [8]:
print("--- Membaca Kembali Berkas Parquet ---")
df_parquet = spark.read.parquet("tugas10_output")
df_parquet.show()

# Menghentikan semua query streaming
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()

--- Membaca Kembali Berkas Parquet ---
+----------+-----------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+---------+---+------------+-------------------+
|       P07|   Cabang_C|Kopi Susu|  2|       15000|2026-11-05 10:05:00|
|       P01|   Cabang_A|Kopi Susu|  2|       15000|2026-11-05 10:00:00|
|       P02|   Cabang_B|Americano|  1|       20000|2026-11-05 10:01:00|
|       P05|   Cabang_B|Kopi Susu|  3|       15000|2026-11-05 10:03:00|
|       P06|   Cabang_A|Americano|  1|       20000|2026-11-05 10:04:00|
|       P03|   Cabang_A|    Latte|  1|       25000|2026-11-05 10:02:00|
|       P04|   Cabang_C| Espresso|  2|       18000|2026-11-05 10:02:30|
+----------+-----------+---------+---+------------+-------------------+

